In [1]:
from google.colab import files
uploaded = files.upload()

Saving twitter.csv to twitter.csv


In [47]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score,f1_score,precision_score,recall_score,roc_auc_score

In [18]:
columns = [
    "target",
    "id",
    "date",
    "flag",
    "user",
    "text"
]
df = pd.read_csv("twitter.csv", encoding='latin1',names=columns,header=None)
print(df.shape)
print(df.columns)
df.head()

(1600000, 6)
Index(['target', 'id', 'date', 'flag', 'user', 'text'], dtype='object')


,target,id,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [19]:
print(df.describe(include="all"))

              target            id  ...      user                       text
count   1.600000e+06  1.600000e+06  ...   1600000                    1600000
unique           NaN           NaN  ...    659775                    1581466
top              NaN           NaN  ...  lost_dog  isPlayer Has Died! Sorry 
freq             NaN           NaN  ...       549                        210
mean    2.000000e+00  1.998818e+09  ...       NaN                        NaN
std     2.000001e+00  1.935761e+08  ...       NaN                        NaN
min     0.000000e+00  1.467810e+09  ...       NaN                        NaN
25%     0.000000e+00  1.956916e+09  ...       NaN                        NaN
50%     2.000000e+00  2.002102e+09  ...       NaN                        NaN
75%     4.000000e+00  2.177059e+09  ...       NaN                        NaN
max     4.000000e+00  2.329206e+09  ...       NaN                        NaN

[11 rows x 6 columns]


In [20]:
print("Missing values:")
print(df.isnull().sum())

Missing values:
target    0
id        0
date      0
flag      0
user      0
text      0
dtype: int64


In [21]:
print("Duplicate rows:", df.duplicated().sum())

print("Duplicate tweets:", df["text"].duplicated().sum())

Duplicate rows: 0
Duplicate tweets: 18534


In [22]:
print("Sentiment distribution:")
print(df["target"].value_counts())

Sentiment distribution:
target
0    800000
4    800000
Name: count, dtype: int64


In [24]:
print("Unique users:", df["user"].nunique())
print("Total tweets:", len(df))

Unique users: 659775
Total tweets: 1600000


In [25]:
import re

def clean_tweet(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\brt\b", " ", text)
    text = re.sub(r"[^a-z0-9#'\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [26]:
df["clean_text"] = df["text"].apply(clean_tweet)

In [27]:
df.head()

,target,id,date,flag,user,text,clean_text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",a that's a bummer you shoulda got david carr o...
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...,is upset that he can't update his facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...,i dived many times for the ball managed to sav...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all....",no it's not behaving at all i'm mad why am i h...


In [28]:
print("Duplicates before:", df["clean_text"].duplicated().sum())

df = df.drop_duplicates(
    subset="clean_text"
).reset_index(drop=True)

print("Shape after removing duplicates:", df.shape)

Duplicates before: 79053
Shape after removing duplicates: (1520947, 7)


In [29]:
df["sentiment"] = df["target"].map({
    0: 0,
    4: 1
})

In [30]:
x = df["clean_text"]
y = df["target"]
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify = y)

In [31]:
x_train = x_train.reset_index(drop=True)
x_test = x_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [32]:
tfidf = TfidfVectorizer(max_features=10000,ngram_range=(1,3),min_df=3,max_df=0.95,sublinear_tf=True)
x_train = tfidf.fit_transform(x_train)
x_test = tfidf.transform(x_test)

In [35]:
features = tfidf.get_feature_names_out()

print("First 50 features:")
print(features[:50])

First 50 features:
['00' '000' '09' '10' '10 days' '10 minutes' '100' '1000' '10am' '10pm'
 '11' '12' '12 hours' '13' '14' '140' '15' '15 minutes' '16' '17' '17th'
 '18' '18th' '19' '1st' '20' '20 minutes' '200' '2009' '2010' '21' '21st'
 '22' '23' '24' '24 hours' '25' '26' '27' '28' '29' '2am' '2day' '2moro'
 '2morrow' '2nd' '2night' '2nite' '30' '30 am']


In [40]:
logistic_model = LogisticRegression(
    C=1.0,
    max_iter=100,
    solver="liblinear",
    random_state=42
)
logistic_model.fit(x_train, y_train)

LogisticRegression(random_state=42, solver='liblinear')

In [41]:
y_pred_lr = logistic_model.predict(x_test)

y_prob_lr = logistic_model.predict_proba(
    x_test
)[:, 1]

In [45]:
accuracy_lr = accuracy_score(y_test, y_pred_lr)

precision_lr = precision_score(
    y_test,
    y_pred_lr,
    pos_label=4
)

recall_lr = recall_score(
    y_test,
    y_pred_lr,
    pos_label=4
)

f1_lr = f1_score(
    y_test,
    y_pred_lr,
    pos_label=4
)

roc_auc_lr = roc_auc_score(
    y_test,
    y_prob_lr
)

print("Logistic Regression Results")
print("---------------------------")
print(f"Accuracy : {accuracy_lr:.4f}")
print(f"Precision: {precision_lr:.4f}")
print(f"Recall   : {recall_lr:.4f}")
print(f"F1 Score : {f1_lr:.4f}")
print(f"ROC-AUC  : {roc_auc_lr:.4f}")

Logistic Regression Results
---------------------------
Accuracy : 0.8014
Precision: 0.7923
Recall   : 0.8120
F1 Score : 0.8020
ROC-AUC  : 0.8811


In [46]:
print(
    classification_report(
        y_test,
        y_pred_lr,
        target_names=[
            "Negative",
            "Positive"
        ]
    )
)

              precision    recall  f1-score   support

    Negative       0.81      0.79      0.80    153507
    Positive       0.79      0.81      0.80    150683

    accuracy                           0.80    304190
   macro avg       0.80      0.80      0.80    304190
weighted avg       0.80      0.80      0.80    304190



In [48]:
nb_model = MultinomialNB(
    alpha=1.0,
    fit_prior=True,
    class_prior=None
)
nb_model.fit(x_train, y_train)

MultinomialNB()

In [49]:
nb_pred = nb_model.predict(x_test)

nb_prob = nb_model.predict_proba(
    x_test
)[:, 1]

In [51]:
accuracy_nb = accuracy_score(y_test,nb_pred)
precision_nb = precision_score(y_test,nb_pred,pos_label=4)
recall_nb = recall_score(y_test,nb_pred,pos_label=4)
f1_nb = f1_score(y_test,nb_pred,pos_label=4)
roc_auc_nb = roc_auc_score(y_test,nb_prob)
print("Naive Bayes Results")
print("-------------------")
print(f"Accuracy : {accuracy_nb:.4f}")
print(f"Precision: {precision_nb:.4f}")
print(f"Recall   : {recall_nb:.4f}")
print(f"F1 Score : {f1_nb:.4f}")
print(f"ROC-AUC  : {roc_auc_nb:.4f}")

Naive Bayes Results
-------------------
Accuracy : 0.7750
Precision: 0.7783
Recall   : 0.7631
F1 Score : 0.7706
ROC-AUC  : 0.8556


In [52]:
print(classification_report(y_test,nb_pred,target_names=["Negative","Positive"]))

              precision    recall  f1-score   support

    Negative       0.77      0.79      0.78    153507
    Positive       0.78      0.76      0.77    150683

    accuracy                           0.77    304190
   macro avg       0.78      0.77      0.77    304190
weighted avg       0.78      0.77      0.77    304190



In [53]:
print("Model Comparison")
print("="*35)
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Naive Bayes"],
    "Accuracy": [accuracy_lr, accuracy_nb],
    "Precision": [precision_lr, precision_nb],
    "Recall": [recall_lr, recall_nb],
    "F1 Score": [f1_lr, f1_nb],
    "ROC-AUC": [roc_auc_lr, roc_auc_nb]
})
print(results)

Model Comparison
                 Model  Accuracy  Precision    Recall  F1 Score   ROC-AUC
0  Logistic Regression  0.801394   0.792267  0.811963  0.801994  0.881139
1          Naive Bayes  0.774993   0.778331  0.763099  0.770640  0.855579
